In [9]:
from pathlib import Path
import pandas as pd

DATA_PATH = (
    Path("../data/processed")
    / "cic_ids2017_unified.csv"
)

CHUNK_SIZE = 100_000

chunk = pd.read_csv(
    DATA_PATH,
    nrows=CHUNK_SIZE
)

print("Chunk loaded successfully")
print("Shape:", chunk.shape)
print("Memory usage:", chunk.memory_usage(deep=True).sum() / 1024**2, "MB")

Chunk loaded successfully
Shape: (100000, 79)
Memory usage: 60.726444244384766 MB


In [10]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = (
    Path("../data/processed")
    / "cic_ids2017_unified.csv"
)

ML_OUTPUT_PATH = (
    Path("../data/processed")
    / "ml_dataset.csv"
)

BINARY_OUTPUT_PATH = (
    Path("../data/processed")
    / "ml_binary_dataset.csv"
)

CHUNK_SIZE = 100_000

print("Ready for chunk processing.")

Ready for chunk processing.


In [11]:
sample = pd.read_csv(
    DATA_PATH,
    nrows=10_000
)

print("Total columns:", len(sample.columns))

for i, column in enumerate(sample.columns):
    print(i, column)

Total columns: 79
0 Destination_Port
1 Flow_Duration
2 Total_Fwd_Packets
3 Total_Backward_Packets
4 Total_Length_of_Fwd_Packets
5 Total_Length_of_Bwd_Packets
6 Fwd_Packet_Length_Max
7 Fwd_Packet_Length_Min
8 Fwd_Packet_Length_Mean
9 Fwd_Packet_Length_Std
10 Bwd_Packet_Length_Max
11 Bwd_Packet_Length_Min
12 Bwd_Packet_Length_Mean
13 Bwd_Packet_Length_Std
14 Flow_Bytes_per_s
15 Flow_Packets_per_s
16 Flow_IAT_Mean
17 Flow_IAT_Std
18 Flow_IAT_Max
19 Flow_IAT_Min
20 Fwd_IAT_Total
21 Fwd_IAT_Mean
22 Fwd_IAT_Std
23 Fwd_IAT_Max
24 Fwd_IAT_Min
25 Bwd_IAT_Total
26 Bwd_IAT_Mean
27 Bwd_IAT_Std
28 Bwd_IAT_Max
29 Bwd_IAT_Min
30 Fwd_PSH_Flags
31 Bwd_PSH_Flags
32 Fwd_URG_Flags
33 Bwd_URG_Flags
34 Fwd_Header_Length
35 Bwd_Header_Length
36 Fwd_Packets_per_s
37 Bwd_Packets_per_s
38 Min_Packet_Length
39 Max_Packet_Length
40 Packet_Length_Mean
41 Packet_Length_Std
42 Packet_Length_Variance
43 FIN_Flag_Count
44 SYN_Flag_Count
45 RST_Flag_Count
46 PSH_Flag_Count
47 ACK_Flag_Count
48 URG_Flag_Count
49 CWE_Fla

In [12]:
TARGET_COLUMN = "Label"

EXCLUDED_FROM_ML = []

for column in sample.columns:

    name = column.lower()

    if column == TARGET_COLUMN:
        continue

    if any(
        keyword in name
        for keyword in [
            "source_ip",
            "destination_ip",
            "timestamp"
        ]
    ):
        EXCLUDED_FROM_ML.append(column)

print("Excluded from ML:")

for column in EXCLUDED_FROM_ML:
    print("-", column)

Excluded from ML:


In [13]:
ML_FEATURES = [
    column
    for column in sample.columns
    if column not in EXCLUDED_FROM_ML
    and column != TARGET_COLUMN
]

print("\nNumber of ML features:", len(ML_FEATURES))


Number of ML features: 78


In [14]:
non_numeric = sample[ML_FEATURES].select_dtypes(
    exclude=np.number
).columns.tolist()

print("Non-numeric ML features:")

for column in non_numeric:
    print("-", column)

Non-numeric ML features:


In [16]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = (
    Path("../data/processed")
    / "cic_ids2017_unified.csv"
)

ML_OUTPUT_PATH = (
    Path("../data/processed")
    / "ml_dataset.csv"
)

BINARY_OUTPUT_PATH = (
    Path("../data/processed")
    / "ml_binary_dataset.csv"
)

CHUNK_SIZE = 100_000

TARGET_COLUMN = "Label"

print("Starting full dataset processing...")

Starting full dataset processing...


In [17]:
sample = pd.read_csv(
    DATA_PATH,
    nrows=10_000
)

ML_FEATURES = [
    column
    for column in sample.columns
    if column != TARGET_COLUMN
]

print("Total ML features:", len(ML_FEATURES))

Total ML features: 78


In [18]:
unique_values = {
    column: set()
    for column in ML_FEATURES
}

chunk_number = 0

for chunk in pd.read_csv(
    DATA_PATH,
    chunksize=CHUNK_SIZE
):

    chunk_number += 1

    for column in ML_FEATURES:
        values = chunk[column].dropna().unique()

        if len(unique_values[column]) <= 2:
            unique_values[column].update(values)

    if chunk_number % 10 == 0:
        print(
            f"Processed {chunk_number} chunks..."
        )

constant_features = [
    column
    for column in ML_FEATURES
    if len(unique_values[column]) <= 1
]

print("\nConstant features:")
print(constant_features)

print(
    "\nNumber of constant features:",
    len(constant_features)
)

Processed 10 chunks...
Processed 20 chunks...

Constant features:
['Bwd_PSH_Flags', 'Bwd_URG_Flags', 'Fwd_Avg_Bytes_per_Bulk', 'Fwd_Avg_Packets_per_Bulk', 'Fwd_Avg_Bulk_Rate', 'Bwd_Avg_Bytes_per_Bulk', 'Bwd_Avg_Packets_per_Bulk', 'Bwd_Avg_Bulk_Rate']

Number of constant features: 8


In [19]:
FINAL_ML_FEATURES = [
    column
    for column in ML_FEATURES
    if column not in constant_features
]

print(
    "Final ML feature count:",
    len(FINAL_ML_FEATURES)
)

Final ML feature count: 70


In [20]:
# Remove old output files if they exist
if ML_OUTPUT_PATH.exists():
    ML_OUTPUT_PATH.unlink()

if BINARY_OUTPUT_PATH.exists():
    BINARY_OUTPUT_PATH.unlink()


first_ml_chunk = True
first_binary_chunk = True

total_rows = 0

for chunk_number, chunk in enumerate(
    pd.read_csv(
        DATA_PATH,
        chunksize=CHUNK_SIZE
    ),
    start=1
):

    # -----------------------------
    # Clean labels
    # -----------------------------

    chunk[TARGET_COLUMN] = (
        chunk[TARGET_COLUMN]
        .astype(str)
        .str.strip()
    )

    # -----------------------------
    # Select ML features
    # -----------------------------

    X_chunk = chunk[
        FINAL_ML_FEATURES
    ].copy()

    # -----------------------------
    # Handle infinite values
    # -----------------------------

    X_chunk = X_chunk.replace(
        [np.inf, -np.inf],
        np.nan
    )

    # -----------------------------
    # Handle missing values
    # -----------------------------

    for column in X_chunk.columns:

        if X_chunk[column].isnull().any():

            median_value = X_chunk[column].median()

            X_chunk[column] = (
                X_chunk[column]
                .fillna(median_value)
            )

    # -----------------------------
    # Multi-class dataset
    # -----------------------------

    ml_chunk = X_chunk.copy()

    ml_chunk["Label"] = (
        chunk[TARGET_COLUMN]
        .values
    )

    ml_chunk.to_csv(
        ML_OUTPUT_PATH,
        mode="w" if first_ml_chunk else "a",
        header=first_ml_chunk,
        index=False
    )

    first_ml_chunk = False

    # -----------------------------
    # Binary dataset
    # -----------------------------

    binary_chunk = X_chunk.copy()

    binary_chunk["Attack"] = (
        chunk[TARGET_COLUMN]
        .str.upper()
        .ne("BENIGN")
        .astype(int)
        .values
    )

    binary_chunk.to_csv(
        BINARY_OUTPUT_PATH,
        mode="w" if first_binary_chunk else "a",
        header=first_binary_chunk,
        index=False
    )

    first_binary_chunk = False

    total_rows += len(chunk)

    if chunk_number % 5 == 0:
        print(
            f"Processed {total_rows:,} rows..."
        )

print("\nProcessing complete!")

print(
    "Total rows processed:",
    f"{total_rows:,}"
)

Processed 500,000 rows...
Processed 1,000,000 rows...
Processed 1,500,000 rows...
Processed 2,000,000 rows...
Processed 2,500,000 rows...

Processing complete!
Total rows processed: 2,574,264


In [21]:
print(
    "ML dataset exists:",
    ML_OUTPUT_PATH.exists()
)

print(
    "Binary dataset exists:",
    BINARY_OUTPUT_PATH.exists()
)

ML dataset exists: True
Binary dataset exists: True


In [22]:
print(
    "ML dataset size:",
    ML_OUTPUT_PATH.stat().st_size / (1024**3),
    "GB"
)

print(
    "Binary dataset size:",
    BINARY_OUTPUT_PATH.stat().st_size / (1024**3),
    "GB"
)

ML dataset size: 0.7980794459581375 GB
Binary dataset size: 0.7856160709634423 GB


In [23]:
ml_check = pd.read_csv(
    ML_OUTPUT_PATH,
    nrows=10_000
)

binary_check = pd.read_csv(
    BINARY_OUTPUT_PATH,
    nrows=10_000
)

print(
    "ML sample shape:",
    ml_check.shape
)

print(
    "Binary sample shape:",
    binary_check.shape
)

print(
    ml_check["Label"].value_counts()
)

print(
    binary_check["Attack"].value_counts()
)

ML sample shape: (10000, 71)
Binary sample shape: (10000, 71)
Label
BENIGN    10000
Name: count, dtype: int64
Attack
0    10000
Name: count, dtype: int64


#####

In [24]:
from collections import Counter

label_counts = Counter()
attack_counts = Counter()

for chunk_number, chunk in enumerate(
    pd.read_csv(
        DATA_PATH,
        chunksize=CHUNK_SIZE,
        usecols=["Label"]
    ),
    start=1
):

    labels = (
        chunk["Label"]
        .astype(str)
        .str.strip()
    )

    label_counts.update(labels)

    attack_counts.update(
        labels.str.upper().ne("BENIGN").astype(int)
    )

    if chunk_number % 10 == 0:
        print(
            f"Scanned {chunk_number} chunks..."
        )

print("\nComplete Multi-class Distribution:")

for label, count in label_counts.most_common():
    print(
        f"{label}: {count:,}"
    )

print("\nComplete Binary Distribution:")

print(
    "BENIGN:",
    attack_counts[0]
)

print(
    "ATTACK:",
    attack_counts[1]
)

Scanned 10 chunks...
Scanned 20 chunks...

Complete Multi-class Distribution:
BENIGN: 2,148,386
DoS Hulk: 172,849
DDoS: 128,016
PortScan: 90,819
DoS GoldenEye: 10,286
FTP-Patator: 5,933
DoS slowloris: 5,385
DoS Slowhttptest: 5,228
SSH-Patator: 3,219
Bot: 1,953
Web Attack � Brute Force: 1,470
Web Attack � XSS: 652
Infiltration: 36
Web Attack � Sql Injection: 21
Heartbleed: 11

Complete Binary Distribution:
BENIGN: 2148386
ATTACK: 425878


In [25]:
ml_label_counts = Counter()

for chunk in pd.read_csv(
    ML_OUTPUT_PATH,
    chunksize=CHUNK_SIZE,
    usecols=["Label"]
):

    labels = (
        chunk["Label"]
        .astype(str)
        .str.strip()
    )

    ml_label_counts.update(labels)

print("\nML Dataset Label Distribution:")

for label, count in ml_label_counts.most_common():
    print(
        f"{label}: {count:,}"
    )


ML Dataset Label Distribution:
BENIGN: 2,148,386
DoS Hulk: 172,849
DDoS: 128,016
PortScan: 90,819
DoS GoldenEye: 10,286
FTP-Patator: 5,933
DoS slowloris: 5,385
DoS Slowhttptest: 5,228
SSH-Patator: 3,219
Bot: 1,953
Web Attack � Brute Force: 1,470
Web Attack � XSS: 652
Infiltration: 36
Web Attack � Sql Injection: 21
Heartbleed: 11


In [26]:
binary_counts = Counter()

for chunk in pd.read_csv(
    BINARY_OUTPUT_PATH,
    chunksize=CHUNK_SIZE,
    usecols=["Attack"]
):

    binary_counts.update(
        chunk["Attack"]
    )

print("\nBinary Dataset Distribution:")

print(
    "BENIGN (0):",
    binary_counts[0]
)

print(
    "ATTACK (1):",
    binary_counts[1]
)


Binary Dataset Distribution:
BENIGN (0): 2148386
ATTACK (1): 425878


In [ ]:
print(
    "\nOriginal rows:",
    total_rows
)

print(
    "ML dataset rows:",
    sum(ml_label_counts.values())
)

print(
    "Binary dataset rows:",
    sum(binary_counts.values())
)


Original rows: 2574264
ML dataset rows: 2574264
Binary dataset rows: 2574264


: 